# Day 2 - Reinforcement Learning, Agency, Minecraft Agents and MARL

These are study notes for Day 2 of **Artificial Intelligence for Computer Games**.

The notebook connects five threads:

- reinforcement learning fundamentals and DQN practice,
- RL for large language models and AI agency,
- Minecraft as a long-horizon embodied-agent environment,
- multi-agent reinforcement learning,
- the shared problems of credit assignment, exploration, and evaluation.

The notebook is intentionally Markdown-heavy. Code cells are small conceptual examples, not full training scripts.


## How To Read This Bilingual Notebook / 如何读这份双语笔记

Format:

- **English** keeps the professional classroom wording.
- **中文解释** lowers the entry barrier and explains what the term means in plain language.
- **Example** shows how the concept appears in Boxing, Breakout, Minecraft, or MARL.
- **Common mistake** points out where beginners usually get confused.

小白读法：

1. 先读中文解释，知道“这东西在干嘛”。
2. 再回头看英文术语，因为考试、slides、paper 都会用英文。
3. 最后看 example，把抽象概念绑到游戏场景里。

Professional target:

You should be able to explain each concept in English, but also understand the intuition deeply enough to implement or debug code.


## Source Discipline

Primary local sources used:

| Source | Role |
| --- | --- |
| `day2.pdf` | Technical survey on RL techniques for LLMs: PPO, Q-learning, GRPO, RLHF, RLAIF, Constitutional AI, DPO, UNA, outcome/process rewards, verifier-guided RL, RLVR, Goodharting, and dynamic evaluation. |
| `Architecting_AI_Agency day2.pdf` | Lecture framing: statistical mimicry -> goal-directed behavior, next-token prediction -> trajectory optimization, alignment-optimization gap, PPO vs GRPO, desirability vs validity, measurement crisis, dynamic compute. |
| `Multi-agent reinforcement learning day2.pdf` | Ville Hautamaki's MARL lecture: motivation, joint actions, level-based foraging, MARL dimensions, game-model hierarchy, POSG, Dec-POMDP, best response, minimax, Nash equilibrium. |
| Live Day 2 classroom notes | Minecraft timeline, DQN blackboard loop, AI agency examples, and instructor emphasis. |
| `Hautamaki-lab/Summer-School-2026` tutorial context | PettingZoo / Atari practice, Boxing project, and DQN setup. |

**Source note:** when a point comes directly from the lecture framing, this notebook keeps the lecture terminology.  
**Clarification:** when a standard textbook detail is added to make the concept easier to study, it is explicitly labelled as clarification.


## Beginner Glossary I - RL and DQN / 入门词汇 I：强化学习与 DQN

Use this table as a dictionary while reading. The English word is the classroom/professional term; the Chinese column explains it in plain language.

| Term | 中文注释 | Beginner meaning / 小白解释 | Example |
| --- | --- | --- | --- |
| agent | 智能体 | The learner/decision maker. 会做动作的“玩家”。 | Boxing player controlled by code. |
| environment | 环境 | The world that reacts to the agent. 动作发生的世界。 | Atari Boxing game. |
| state, $s$ | 状态 | Complete situation used for decision making. 理想情况下“完整局面”。 | Positions, scores, timers, velocities. |
| observation, $o$ | 观测 | What the agent actually sees. 真实能看到的信息，可能不完整。 | Raw screen pixels. |
| action, $a$ | 动作 | A choice made by the agent. | move left, punch, noop. |
| reward, $r$ | 奖励 | Immediate score signal after an action. 单步反馈。 | +1 for scoring a hit. |
| return, $G$ | 回报 | Future rewards added together, often discounted. 长期总收益。 | Winning after many good moves. |
| trajectory | 轨迹 | A full sequence of states, actions, rewards. 一整段游戏经历。 | one episode of Breakout. |
| episode | 回合 | One run from reset to done. 从开始到结束的一局。 | one Boxing match. |
| policy, $\pi$ | 策略 | Rule/model that chooses actions. | epsilon-greedy action choice. |
| deterministic policy | 确定性策略 | Same state gives same action. | always choose highest Q action. |
| stochastic policy | 随机策略 | Chooses actions with probabilities. | 10% random action. |
| discount factor, $\gamma$ | 折扣因子 | How much future reward matters. 越接近 1 越重视未来。 | $\gamma=0.99$ for long-term game reward. |
| value function, $V(s)$ | 状态价值 | How good a state is. | "Being near the ball is good." |
| Q-function, $Q(s,a)$ | 动作价值 | How good an action is in a state. | "Punch now is worth 0.8." |
| MDP | 马尔可夫决策过程 | Formal model for single-agent sequential decision making. | simplified Atari game. |
| Markov property | 马尔可夫性质 | Current state is enough; old history is not needed if state is complete. | If full board state is known, earlier moves need not be stored. |
| transition probability, $P(s' \mid s,a)$ | 状态转移概率 | Probability of next state after action. | punch may lead to opponent moving back. |
| terminal state | 终止状态 | End of episode. | game over. |
| Q-learning | Q 学习 | Learn action values from reward plus future value. | update punch/dodge values. |
| Bellman target | 贝尔曼目标 | What Q-value should move toward. | $r+\gamma\max Q(s',a')$. |
| DQN | 深度 Q 网络 | Neural network version of Q-learning. | pixels -> Q-values for actions. |
| replay buffer | 经验回放池 | Stores old transitions for later training. | memory of past game steps. |
| mini-batch | 小批量 | Random small set sampled for one update. | 32 transitions. |
| epsilon-greedy | ε-贪心 | Mostly choose best action, sometimes explore randomly. | 90% best action, 10% random. |
| loss | 损失函数 | Number measuring prediction error. 越小越好。 | MSE between target and Q estimate. |
| MSE | 均方误差 | Average squared error. | $(target-prediction)^2$. |
| optimizer | 优化器 | Algorithm that updates neural network weights. | Adam. |
| backpropagation | 反向传播 | Computes gradients through a neural network. | loss -> parameter gradients. |
| target network | 目标网络 | Slower copy used to stabilize DQN targets. | update every 1000 steps. |


## Beginner Glossary II - LLM Agency and Minecraft / 入门词汇 II：LLM Agency 与 Minecraft

| Term | 中文注释 | Beginner meaning / 小白解释 | Example |
| --- | --- | --- | --- |
| pretraining | 预训练 | Large-scale initial training, usually before task-specific tuning. | next-token prediction on text. |
| next-token prediction | 下一个 token 预测 | Predict the next word/subword. | "The cat sat on the ..." -> "mat". |
| token | 文本单位 | A chunk of text used by language models. | word, subword, punctuation. |
| sequence-level objective | 序列级目标 | Judge the whole answer, not each token separately. | full solution correctness. |
| trajectory optimization | 轨迹优化 | Optimize a whole action/answer path over time. | multi-step reasoning answer. |
| RLHF | 人类反馈强化学习 | Learn from human preference rankings. | humans choose better answer. |
| RLAIF | AI 反馈强化学习 | AI evaluator replaces/augments human judge. | model judges two outputs. |
| reward model | 奖励模型 | Model that predicts preference score. | score helpfulness. |
| reference model | 参考模型 | Base model used to prevent excessive policy drift. | original SFT model. |
| critic/value model | 价值/评论家模型 | Estimates future value for policy optimization. | PPO value head. |
| PPO | 近端策略优化 | Conservative policy update method. | clipped probability ratio. |
| KL constraint | KL 约束 | Keeps new policy close to old/reference policy. | avoid destructive drift. |
| GRPO | 组相对策略优化 | Scores a group of candidate outputs and uses relative advantage. | compare 8 sampled solutions. |
| DPO | 直接偏好优化 | Learns directly from preferred/rejected pairs without explicit reward model. | chosen vs rejected answer. |
| UNA | unified feedback method | Unifies pairwise, binary, and scalar feedback. | combine thumbs-up and rankings. |
| outcome supervision | 结果监督 | Reward only final answer/result. | unit test pass/fail. |
| process supervision | 过程监督 | Reward intermediate reasoning steps. | grade each math step. |
| verifier | 验证器 | System that checks correctness. | compiler, tests, proof checker. |
| RLVR | 可验证奖励强化学习 | Uses objective verifiable rewards. | unit tests for code. |
| proxy objective | 代理目标 | Measurable substitute for true goal. | reward score instead of real usefulness. |
| Goodhart's Law | 古德哈特定律 | Once a measure becomes a target, it can stop measuring the real thing. | model optimizes judge quirks. |
| reward hacking | 奖励黑客/钻奖励漏洞 | Agent exploits reward without doing intended task. | high score through boring exploit. |
| behavioral cloning | 行为克隆 | Supervised imitation of expert actions. | learn from Minecraft videos. |
| inverse dynamics model, IDM | 逆动力学模型 | Predicts action from two observations. | $(o_t,o_{t+1}) -> a_t$. |
| skill library | 技能库 | Stored reusable procedures. | "craft table" program. |
| procedural memory | 程序性记忆 | Knowing how to do something. | reusable Mineflayer code. |
| declarative memory | 陈述性记忆 | Knowing facts. | recipe knowledge. |
| vector database | 向量数据库 | Search memory by semantic similarity. | retrieve similar subgoal solution. |
| VLM | 视觉语言模型 | Understand/evaluate image plus text. | judge whether build looks like a house. |
| VLA | 视觉-语言-动作模型 | Uses vision and language to act. | see screen, read goal, press controls. |
| temporal abstraction | 时间抽象 | One high-level action represents many low-level steps. | generated program mines wood. |


## Beginner Glossary III - MARL and Game Theory / 入门词汇 III：多智能体与博弈论

| Term | 中文注释 | Beginner meaning / 小白解释 | Example |
| --- | --- | --- | --- |
| MARL | 多智能体强化学习 | RL with more than one decision-making agent. | two Boxing players. |
| joint action | 联合动作 | All agents' actions together. | $(a_1,a_2,a_3)$. |
| centralized | 集中式 | One controller or training process sees/controls more. | one super-agent controls 3 robots. |
| decentralized | 分布式/去中心化 | Each agent acts separately with local info. | each robot has own policy. |
| communication | 通信 | Agents share messages/information. | teammate sends location. |
| observability | 可观测性 | What agents can see. | full map vs fog of war. |
| partial observability | 部分可观测 | Agent sees incomplete/noisy info. | card game hidden cards. |
| zero-sum game | 零和博弈 | One agent's gain is another's loss. | chess-style win/loss. |
| common-reward game | 共同奖励博弈 | All agents receive same reward. | team foraging. |
| general-sum game | 一般和博弈 | Rewards can mix cooperation and competition. | Diplomacy. |
| normal-form game | 标准式博弈 | One-shot interaction with payoff matrix. | rock-paper-scissors. |
| repeated game | 重复博弈 | Same one-shot game repeated over rounds. | repeated prisoner's dilemma. |
| stochastic game | 随机/马尔可夫博弈 | Multi-agent MDP with environment state. | multi-player gridworld. |
| POSG | 部分可观测随机博弈 | Multi-agent stochastic game with partial observations. | RTS fog of war. |
| Dec-POMDP | 分散式 POMDP | Common-reward POSG. | robot team with shared reward. |
| solution concept | 解概念 | Definition of what counts as a solution. | Nash equilibrium. |
| expected return | 期望回报 | Average long-term reward under policies. | expected tournament score. |
| best response | 最优反应 | Best policy when others are fixed. | counter a known opponent. |
| minimax | 极小极大 | Optimize against worst-case opponent in zero-sum games. | robust chess strategy. |
| Nash equilibrium | 纳什均衡 | No agent can improve by changing alone. | everyone is best-responding. |
| non-stationarity | 非平稳性 | Environment changes because other agents learn too. | opponent strategy keeps changing. |
| credit assignment | 归因问题 | Which action/agent caused success or failure? | which teammate earned team reward? |


## 0. Day 2 Map

Day 2 is not a list of unrelated topics. It is a progression from simple sequential decisions to rich agentic behavior:

```text
RL fundamentals
    -> Q-learning and DQN
    -> RL for LLMs
    -> alignment and reasoning objectives
    -> Minecraft agents
    -> multi-agent reinforcement learning
    -> human-AI collaboration
```

A compact map:

| Layer | Question | Representative idea |
| --- | --- | --- |
| Classical RL | Which action maximizes future reward? | MDP, policy, value, Q-function |
| DQN | How do we learn Q-values from high-dimensional observations? | neural Q-network + replay buffer |
| RL for LLMs | How do we optimize generated sequences, not just next-token likelihood? | PPO, GRPO, RLHF, DPO, RLVR |
| Agency | How do models plan, use tools, and revise actions over time? | hierarchy, tool use, program synthesis |
| Minecraft | How do agents handle long horizons, open worlds, language goals, and embodiment? | VPT, MineRL, Voyager-style code actions, VLA |
| MARL | What changes when many agents act together? | joint action, stochastic games, POSG, Nash equilibrium |

The recurring theme is:

> Intelligence in games and agents is less about one isolated prediction and more about choosing actions whose consequences unfold over time.


### 中文导读 / Beginner Notes

Day 2 的核心不是“背很多算法名字”，而是看一个演化过程：

```text
单个 agent 学会玩游戏
    -> 多个 agent 一起/对抗
    -> LLM 不只预测文字，而是做长期任务
    -> Minecraft 用开放世界测试 agent 能力
```

**Example:** Boxing 里一个 DQN agent 只需要决定自己下一步动作；MARL 里还要考虑对手也在决策；Minecraft 里 agent 甚至要先计划“砍树 -> 合成工具 -> 挖矿”。


## Evolution Thread - Why These Ideas Appeared / 演化主线：为什么这些方法会出现

This is the "big picture" thread for beginners who want the logic behind the topic order.

### 1. From MDP to Q-learning

Problem:

- We need a formal way to describe sequential decision making.
- An action can be bad immediately but good later.

Evolution:

```text
MDP gives the language
    -> Q-learning gives a way to learn action values
```

Example:

- In Boxing, moving backward may get no immediate reward.
- But it can avoid a punch and set up a better attack later.

### 2. From Q-learning to DQN

Problem:

- A Q-table works for small state spaces.
- Atari observations are images, so the state space is huge.

Evolution:

```text
Q-table
    -> neural network approximator
    -> DQN
```

Example:

- Instead of storing `Q(pixel_image, action)` in a table, DQN learns a function:

$$
Q_\theta(\text{image}, a)
$$

### 3. From DQN/PPO to RL for LLMs

Problem:

- LLM pretraining predicts the next token.
- Users care about the full answer or full task outcome.

Evolution:

```text
token likelihood
    -> sequence-level reward
    -> RLHF / PPO / DPO / GRPO / RLVR
```

Example:

- A generated proof can sound fluent but be logically wrong.
- A verifier can judge final correctness more directly.

### 4. From RL for LLMs to agentic tool use

Problem:

- Some tasks require external actions, not just text.

Evolution:

```text
generate text
    -> call tools
    -> generate code
    -> execute programs
    -> observe and revise
```

Example:

- A Minecraft agent writes a Mineflayer program to collect wood, then tests whether it worked.

### 5. From single-agent RL to MARL

Problem:

- Many environments contain other decision makers.
- Other agents can cooperate, compete, communicate, hide information, or learn.

Evolution:

```text
MDP
    -> stochastic game
    -> POSG
    -> equilibrium / best-response concepts
```

Example:

- In Boxing, your best action depends on what the opponent is doing.
- In Diplomacy, cooperation and betrayal both matter.

### 6. From fixed tasks to fuzzy creative tasks

Problem:

- Some goals do not have a clean binary success condition.

Evolution:

```text
fixed reward
    -> human preference
    -> VLM evaluation
    -> dynamic evaluation
```

Example:

- "Mine two logs" is easy to verify.
- "Build a medieval-inspired fortress" needs visual/human judgment.


## 1. RL Refresher

Reinforcement learning studies an agent that repeatedly interacts with an environment:

```text
state/observation -> action -> reward + next state/observation
```

Core terms:

| Term | Meaning | Intuition |
| --- | --- | --- |
| State, $s_t$ | Information that defines the environment at time $t$ | "What situation am I in?" |
| Observation, $o_t$ | What the agent can actually see | May be partial/noisy state information |
| Action, $a_t$ | Choice made by the agent | Move, punch, token, tool call, program |
| Reward, $r_t$ | Immediate scalar feedback | "Was this step good?" |
| Return, $G_t$ | Discounted future reward | "How good was the whole future after this?" |
| Policy, $\pi(a \mid s)$ | Action-selection rule | "What do I do in this state?" |
| Discount, $\gamma$ | Weight on future rewards | Near 1 values long-term outcomes more |
| Value, $V^\pi(s)$ | Expected return from state under policy $\pi$ | "How good is this state?" |
| Q-value, $Q^\pi(s,a)$ | Expected return after taking action $a$ in state $s$ | "How good is this action here?" |

The discounted return is:

$$
G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots
$$

The state-value function is:

$$
V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t=s]
$$

The action-value function is:

$$
Q^\pi(s,a) = \mathbb{E}_\pi[G_t \mid s_t=s, a_t=a]
$$


### 中文导读 / Beginner Notes

强化学习可以想成“打游戏学经验”：

- 看到局面：state / observation
- 做动作：action
- 得分或扣分：reward
- 目标不是眼前一步，而是长期赢：return

**Common mistake:** reward 和 return 不一样。Reward 是“这一秒得了几分”，return 是“从现在开始到未来总共赚多少”。

**Example:** Boxing 里这一拳没得分，但它让你站到更好的位置，未来可能赢。所以只看 immediate reward 不够。


### Markov Decision Process

A Markov Decision Process, or MDP, is the usual formal model for single-agent RL.

One common tuple is:

$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)
$$

where:

- $\mathcal{S}$ is the state space,
- $\mathcal{A}$ is the action space,
- $P(s' \mid s,a)$ is the transition probability,
- $R(s,a,s')$ is the reward function,
- $\gamma \in [0,1]$ is the discount factor.

The Markov property means:

$$
P(s_{t+1} \mid s_t,a_t,s_{t-1},a_{t-1},\ldots) = P(s_{t+1} \mid s_t,a_t)
$$

**Intuition:** the current state contains all relevant information from the past. The agent does not need the entire history if the state is truly Markov.

**Why it matters:** Q-learning and DQN assume the future can be predicted from the current state and action. If the observation is incomplete, the problem is closer to a POMDP.


## 2. Q-learning and DQN

Q-learning learns an estimate of the optimal action-value function:

$$
Q^*(s,a) = \max_\pi Q^\pi(s,a)
$$

The Bellman optimality target is:

$$
y = r + \gamma \max_{a'} Q(s',a')
$$

If the transition is terminal:

$$
y = r
$$

The tabular Q-learning update is:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[r + \gamma \max_{a'}Q(s',a') - Q(s,a)\right]
$$

where $\alpha$ is the learning rate.

**Intuition:** update the current estimate toward a bootstrapped target: immediate reward plus best estimated future value.


### 中文导读 / Beginner Notes

Q-learning 的问题是：

> 在这个状态下，我做这个动作，长期看值不值？

DQN 的升级点是：状态太复杂时，比如 Atari pixel image，不能手写表格，所以用 neural network 近似 $Q(s,a)$。

**Example:** 输入 Breakout 的画面，网络输出 4 个数，分别表示 noop/fire/right/left 的 Q-value。最大那个就是当前看起来最好的动作。

**Common mistake:** DQN 不是直接输出动作，而是输出每个动作的分数。


In [1]:
def bellman_target(reward, gamma, next_q_values, done):
    # Return the one-step Q-learning target.
    if done:
        return reward
    return reward + gamma * max(next_q_values)


print("Non-terminal target:", bellman_target(1.0, 0.99, [0.2, 2.0, 0.7], False))
print("Terminal target:    ", bellman_target(1.0, 0.99, [0.2, 2.0, 0.7], True))


Non-terminal target: 2.98
Terminal target:     1.0


### DQN

Deep Q-Network, or DQN, replaces a table of Q-values with a neural network:

$$
Q_\theta(s,a)
$$

In Atari, the state may be image frames, so a CNN can map pixels to action values.

DQN loss over a mini-batch:

$$
L(\theta) =
\frac{1}{N}\sum_i
\left(y_i - Q_\theta(s_i,a_i)\right)^2
$$

with:

$$
y_i =
\begin{cases}
r_i, & \text{if terminal}\\
r_i + \gamma \max_{a'} Q(s'_i,a'), & \text{otherwise}
\end{cases}
$$

**Clarification:** standard DQN commonly uses a separate target network $Q_{\theta^-}$ for the target term. This stabilizes training by avoiding a moving target that changes every gradient step. This is a standard DQN clarification; do not assume it was present in every blackboard snippet unless shown.


### DQN Blackboard Training Loop

The live DQN sketch can be studied as:

```text
1. Init replay buffer
2. Init policy / Q-network
3. Train in environment

   3.1 choose action using epsilon-greedy policy
   3.2 store transition (o_t, o_{t+1}, a_t, r_t)
   3.3 sample mini-batch from replay buffer
   3.4 train with MSE between Bellman target and Q estimate
```

Transition format:

$$
(s_t, a_t, r_t, s_{t+1}, done)
$$

Epsilon-greedy policy:

```text
with probability epsilon:
    choose random action
otherwise:
    choose argmax_a Q_theta(s_t, a)
```

Why replay buffer matters:

- it decorrelates sequential samples,
- it reuses old experience,
- it makes neural-network optimization less unstable,
- it allows mini-batch training instead of learning only from the newest transition.


In [2]:
import random


def epsilon_greedy(q_values, epsilon, rng=None):
    # Choose random action with probability epsilon; otherwise choose argmax Q.
    rng = rng or random
    if rng.random() < epsilon:
        return rng.randrange(len(q_values)), "explore"
    best_action = max(range(len(q_values)), key=lambda idx: q_values[idx])
    return best_action, "exploit"


rng = random.Random(7)
q_values = [0.1, 0.8, 0.3, 0.2]
for _ in range(6):
    action, mode = epsilon_greedy(q_values, epsilon=0.25, rng=rng)
    print(f"q={q_values} -> action={action}, mode={mode}")


q=[0.1, 0.8, 0.3, 0.2] -> action=1, mode=exploit
q=[0.1, 0.8, 0.3, 0.2] -> action=0, mode=explore
q=[0.1, 0.8, 0.3, 0.2] -> action=0, mode=explore
q=[0.1, 0.8, 0.3, 0.2] -> action=1, mode=exploit
q=[0.1, 0.8, 0.3, 0.2] -> action=1, mode=explore
q=[0.1, 0.8, 0.3, 0.2] -> action=3, mode=explore


In [3]:
from collections import deque
import random


class ReplayBuffer:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.data.append((state, action, reward, next_state, done))

    def sample(self, batch_size, rng=None):
        rng = rng or random
        return rng.sample(list(self.data), batch_size)

    def __len__(self):
        return len(self.data)


memory = ReplayBuffer(capacity=5)
for t in range(7):
    memory.push(f"s{t}", "right", t * 0.5, f"s{t+1}", False)

print("Buffer length:", len(memory))
print("Oldest transitions were overwritten by capacity.")
print("Sample:", memory.sample(2, rng=random.Random(3)))


Buffer length: 5
Oldest transitions were overwritten by capacity.
Sample: [('s3', 'right', 1.5, 's4', False), ('s6', 'right', 3.0, 's7', False)]


### DQN Practice Variables

The class/tutorial context used a Breakout-style example:

```python
env = gym.make("ALE/Breakout-v5", render_mode=None)
agent = BreakoutAgent(env)
memory = ReplayBuffer()
optimizer = torch.optim.Adam(agent.parameters(), ...)

EPISODES = 100
BATCH_SIZE = 32
EPSILON = 0.1
```

Meaning:

| Variable | Meaning |
| --- | --- |
| `env` | Atari environment that returns observations, rewards, termination flags, and info. |
| `agent` | Neural network approximating $Q_\theta(s,a)$. |
| `memory` | Replay buffer storing transitions. |
| `optimizer` | Updates network parameters from loss gradients. |
| `EPISODES` | Number of full environment resets/runs. |
| `BATCH_SIZE` | Number of sampled transitions per gradient update. |
| `EPSILON` | Probability of random exploration. |
| `obs` | Raw observation from the environment. |
| `state` | Processed observation, often tensor/stacked frames. |
| `done` | Whether the episode has ended. |
| `total_reward` | Sum of rewards collected during an episode. |

In our current project setup, the practical direction is Atari Boxing through PettingZoo. The DQN idea is the same, but Boxing is competitive and multi-agent: one learned boxer can be trained against a random, scripted, historical, or self-play opponent.


## 3. From Next-Token Prediction to RL for LLMs

The lecture connected pretrained LLMs and RL through this contrast:

```text
next-token prediction
        ->
trajectory optimization
```

Pretraining:

- LLMs are fundamentally trained by next-token prediction.
- This optimizes statistical likelihood.
- It does not directly optimize long-term correctness, user intent, safety, or task success.

RL formulation for language:

| RL term | LLM interpretation |
| --- | --- |
| State | prompt + previous generated tokens + context |
| Action | next token, tool call, structured output, or program fragment |
| Policy | LLM distribution over next actions |
| Trajectory | full generated response or multi-step agent trace |
| Reward | human preference, AI feedback, verifier result, test result, task success |

Source note: the survey frames RL as a bridge from statistical mimicry toward goal-directed behavior.


### 中文导读 / Beginner Notes

LLM 预训练像是在做“接龙”：给前文，猜下一个 token。  
RL for LLMs 想解决的是：一个回答不是每个 token 看起来顺就行，而是整体要有用、正确、安全、完成任务。

**Example:** 一个数学答案每个句子都像真的，但最后算错了。Next-token likelihood 可能不惩罚它；verifier reward 可以直接看 final answer 是否正确。


## 4. Alignment-Optimization Gap

The AI agency lecture emphasized three gaps:

| Gap | Supervised / pretraining side | RL / agency side |
| --- | --- | --- |
| Objective level | token-level likelihood | sequence-level or trajectory-level objective |
| Signal density | dense supervised labels at many tokens | often sparse reward at final outcome |
| Search behavior | imitate dataset distribution | explore, exploit, and optimize |

The **alignment-optimization gap** is the mismatch between:

- what the model is easy to train on,
- what humans actually want,
- what optimization pressure eventually rewards.

Example:

- A next-token model can produce plausible text.
- A user may want a correct solution, safe answer, or completed task.
- A reward model may imperfectly proxy the user's true goal.

**Why it matters:** the stronger the optimizer, the more important reward design and evaluation become. Optimizing a proxy too hard can reduce true utility.


### 中文导读 / Beginner Notes

Alignment-optimization gap 指：训练时优化的东西，和人真正想要的东西，中间有差距。

**小白版：** 你让模型“拿高分”，但高分规则不完美，模型可能学会钻评分规则漏洞，而不是真正变好。

**Example:** 如果 reward model 偏爱很长的答案，模型可能变得 verbose but vacuous：很长、看起来认真，但没有信息量。


## 5. PPO, Off-policy Q-learning, and GRPO

### PPO

Proximal Policy Optimization is widely used in RLHF-style pipelines because it makes policy updates conservative.

The lecture's PPO components:

- policy model,
- reference model,
- reward model,
- value/critic model.

PPO is stable but computationally heavy.

The clipped probability-ratio idea:

$$
r_t(\theta) =
\frac{\pi_\theta(a_t \mid s_t)}
     {\pi_{\theta_{\text{old}}}(a_t \mid s_t)}
$$

$$
L^{CLIP}(\theta) =
\mathbb{E}_t
\left[
\min\left(
r_t(\theta)\hat{A}_t,
\text{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\hat{A}_t
\right)
\right]
$$

The KL constraint/reference model discourages destructive policy drift away from the base behavior.


### 中文导读 / Beginner Notes

这节是三个方向：

- **PPO:** 稳，更新不要太猛，但模型组件多。
- **Off-policy Q-learning:** 可以用旧数据/静态数据，更省样本，但受 dataset 限制。
- **GRPO:** 不单独训练 critic，而是比较同一组 candidate outputs 的相对好坏。

**Example:** 对同一个 prompt 生成 8 个答案，给每个答案打分。GRPO 不问“绝对好多少”，而看“这 8 个里面谁比平均好”。


### Off-policy Q-learning Approaches

The lecture mentioned off-policy Q-learning approaches such as **ILQL** and **VerifierQ**.

Main idea:

- learn from static datasets,
- improve sample efficiency,
- avoid needing all data from the current policy.

Limitation:

- the learned policy is constrained by the dataset distribution,
- out-of-distribution actions or reasoning paths can be unreliable,
- a static dataset may not contain the exploration needed for a new objective.

Connection to DQN:

- DQN is also off-policy in spirit because the replay buffer stores experiences collected under older behavior policies.
- LLM off-policy RL faces a harder action/state space: tokens, long contexts, and generated trajectories.


### GRPO

Group Relative Policy Optimization removes the separate critic/value model.

The lecture framing:

```text
generate a group of candidate outputs
    -> score all candidates
    -> normalize rewards within the group
    -> estimate relative advantage from group statistics
    -> update policy
```

A simplified group-relative advantage can be written as:

$$
\hat{A}_i =
\frac{r_i - \text{mean}(r_1,\ldots,r_K)}
     {\text{std}(r_1,\ldots,r_K) + \epsilon}
$$

Tradeoff:

| PPO | GRPO |
| --- | --- |
| Uses value/critic model | Removes separate critic/value model |
| More memory and model-management cost | Lower memory cost |
| Stable actor-critic style | Uses group-relative comparisons |
| Fewer samples may be needed per prompt | More sampling/inference may be needed per prompt |

**Why it matters:** GRPO is attractive when critic training is expensive or unstable, especially in large-model settings.


## 6. Alignment / Desirability

The lecture organized one major family of objectives around:

```text
alignment / desirability / preferences
```

Goal:

- match human values,
- follow instructions,
- be helpful,
- be harmless,
- be honest,
- match preferred style.

These objectives often depend on preference feedback rather than one objectively correct answer.


### 中文导读 / Beginner Notes

Alignment/desirability 关心“人喜不喜欢、安不安全、有没有帮助”。它不一定有唯一正确答案。

**Example:** 两个回答都没错，但一个更礼貌、更清楚、更符合用户需求。RLHF/DPO 这类方法就用 preference 来学这种差别。

**Common mistake:** alignment 不等于 factual correctness。一个回答可以很礼貌但事实错误。


### RLHF

Reinforcement Learning from Human Feedback:

```text
pretrained or SFT model
    -> generate responses
    -> human ranking
    -> reward model
    -> PPO optimization
```

Definition:

- humans compare outputs,
- a reward model learns to predict human preference,
- the policy is optimized against that reward model.

Why it matters:

- aligns outputs with human judgments more directly than next-token prediction,
- improves instruction following and preference satisfaction.

Failure mode:

- reward hacking,
- reward-model overoptimization,
- human disagreement,
- expensive data collection.


### RLAIF

Reinforcement Learning from AI Feedback uses AI evaluators instead of, or in addition to, humans.

Benefit:

- cheaper and more scalable feedback.

Important risk from the lecture:

- systematic evaluator bias can be amplified rather than averaged away.

Example failure mode:

- if the AI judge systematically prefers verbose or flattering answers, training may increase verbosity or sycophancy.


### Constitutional AI

Constitutional AI uses explicit rules or principles to guide critique and revision.

Lecture pipeline:

```text
Generation
    -> constitution/rules
    -> self-critique
    -> revision
    -> supervised refinement
    -> RL optimization using constitution-guided feedback
```

Key concepts:

- **critique-revision gap:** the model may correctly identify a problem in its answer but fail to revise it properly.
- **over-refusal:** the model may become too cautious and refuse harmless requests.

Why it matters:

- makes part of the alignment target explicit,
- scales feedback through self-critique or AI critique,
- still depends on the quality and coverage of the constitution.


### DPO

Direct Preference Optimization bypasses an explicit reward model.

Main idea:

- train directly on preferred vs rejected response pairs,
- use a log-ratio objective related to the implicit reward difference,
- simplify the RLHF pipeline.

Benefits:

- simple,
- efficient,
- often more stable than full PPO-style RLHF.

Risks from the lecture:

- length bias,
- noisy preference overfitting,
- limited exploration.


### UNA

UNA was described as interpreting the policy as an implicit reward estimator.

Main idea:

- unify pairwise, binary, and scalar feedback,
- train with supervised regression-style objectives,
- avoid some complexity of explicit reward-model RL.

Benefits:

- flexible,
- efficient,
- can absorb different feedback formats.

Limitation:

- exploration deficit. If training is mostly supervised over feedback data, it may not discover better behaviors outside the dataset.


## 7. Reasoning / Validity

The second major family of objectives is:

```text
reasoning / validity / correctness
```

Goal:

- mathematical correctness,
- logical correctness,
- factual correctness,
- executable correctness,
- real task completion.

This is different from preference alignment. A response may be preferred stylistically but objectively wrong; another may be terse but verifiably correct.


### 中文导读 / Beginner Notes

Reasoning/validity 关心“客观上对不对”。数学、代码、证明、游戏任务完成都属于这一类。

**Example:** Code generation 里，用户不只是想要“看起来像代码”，而是要能通过 unit tests。这里 verifier 比 preference 更重要。


### Outcome-based RL

Outcome-based RL rewards the final answer or final result.

Properties:

- sparse reward,
- scalable if the final outcome is easy to check,
- avoids needing labels for every reasoning step.

Failure modes:

- credit assignment is hard,
- model may learn "right answer, wrong reasoning",
- final correctness may hide fragile intermediate logic.

Connection to games:

- winning or losing a match is an outcome reward,
- but it may not tell which move mattered.


### Process Supervision / CoT Reward Optimization

Process supervision gives feedback to intermediate reasoning steps.

Properties:

- denser reward,
- helps credit assignment,
- can teach better reasoning trajectories.

Costs and risks:

- expensive to label,
- evaluator may be gamed,
- reasoning hacking: producing reasoning that looks good to the evaluator without actually being reliable.


### Verifier-guided RL vs RLVR

Verifier-guided RL:

- policy generates reasoning or actions,
- a separate neural or external verifier scores the result,
- the policy is trained using verifier feedback.

Risk:

- a learned verifier can be exploited.

RLVR, Reinforcement Learning with Verifiable Rewards:

- uses deterministic or objective verifiers,
- examples include compiler, unit tests, symbolic solver, proof checker,
- often gives binary rewards.

Advantages:

- harder to game than learned reward models,
- strong when correctness is objectively checkable.

Limitations:

- sparse reward,
- limited to verifiable domains,
- incomplete tests still allow specification gaming.

**Key distinction:** verifier-guided RL may use a learned proxy. RLVR emphasizes objective/verifiable reward signals.


## 8. Advanced Agency

Advanced agency asks what happens when the "action" is no longer just a primitive button press or token.

Actions can become:

- tool calls,
- API calls,
- code,
- programs,
- subgoals,
- plans,
- communication with another agent or human.


### 中文导读 / Beginner Notes

Advanced agency 的关键变化是：action 不再只是一个 token 或一个按键，而可能是工具调用、API call、一段代码、一个 subgoal。

**Example:** Minecraft agent 的一个 action 可以是“执行一个 Mineflayer program 去砍树”，这背后包含很多低级键鼠动作。


### Debate and Self-play RL

Debate/self-play setup:

```text
proponent
    vs
opponent
    -> judge
    -> learning signal
```

Benefit:

- creates an autocurriculum,
- agents generate increasingly difficult cases for each other,
- useful where external labels are scarce.

Risk:

- sophistry or persuasion instead of truth,
- judge weaknesses become part of the game.

Connection to MARL:

- the learning environment includes other adaptive agents.


### Hierarchical RL / Tool-augmented Reasoning

Hierarchical idea:

```text
high-level manager chooses subgoal/tool
    -> low-level worker executes
    -> manager observes outcome
    -> next subgoal/tool
```

Why useful:

- long-horizon tasks become shorter chunks,
- tools provide external capabilities,
- temporal abstraction reduces decision length.

Failure modes:

- manager/worker credit assignment,
- non-stationarity if worker behavior changes,
- bad subgoals can make good low-level skills fail.


### Program-Synthesis RL

Program synthesis treats code generation as action.

```text
generate code
    -> run compiler/tests/environment
    -> receive execution feedback
    -> revise
```

Why it matters:

- hallucination becomes executable failure,
- unit tests and compilers can provide reward,
- one program can compress many low-level actions.

Limitation:

- test coverage is itself a proxy,
- optimizing against incomplete tests can produce specification gaming.


## 9. Measurement Crisis

The lecture treated measurement as a central problem, not an afterthought.

**Proxy-objective mismatch:**

```text
proxy reward increases
    while
true human utility / truth / correctness plateaus or declines
```

Goodhart's Law:

> When a measure becomes a target, it ceases to be a good measure.

Examples:

- sycophancy,
- verbose but vacuous answers,
- alignment faking,
- exaggerated refusal,
- reasoning hacking,
- unit-test/specification gaming,
- reward hacking in games.

Future direction emphasized by the lecture:

```text
static scalar benchmark
    -> verifier-guided dynamic evaluation
```

Examples of stronger evaluation:

- interactive environments,
- functional verifiers,
- game engines,
- compilers,
- formal proof checkers,
- real task completion.


### 中文导读 / Beginner Notes

Measurement crisis 是 Day 2 很重要的结论：越强的优化器，越容易暴露指标的问题。

**小白版：** 如果考试只考选择题，学生可能学会刷题技巧，但不一定真正理解。模型也一样，会优化 metric，而 metric 不一定等于真实目标。

**Example:** Unit tests 不完整时，代码模型可能写出刚好通过测试但泛化很差的代码。


## 10. Minecraft as an AI Research Environment

Minecraft is valuable because it combines:

- long horizons,
- sparse rewards,
- crafting dependencies,
- exploration,
- partial observability,
- open-ended goals,
- language instructions,
- human creativity,
- collaboration.

This makes it a bridge between games, robotics, planning, and language-based agents.


### 中文导读 / Beginner Notes

Minecraft 为什么重要？因为它不是一个简单小游戏，而是开放世界：

- 要看环境：vision
- 要理解语言目标：language
- 要操作：action
- 要长期计划：planning
- 要复用技能：memory/skills

**Example:** "make a stone pickaxe" 不是一步动作，而是一串依赖任务。


### Chronological Timeline

| Year | Environment / project | Main idea |
| --- | --- | --- |
| 2016 | Project Malmo | Early Minecraft AI/RL environment with Python interface. |
| 2019 | MineRL | More Minecraft-like long-horizon tasks and item dependencies. |
| 2019 | CraftAssist | Human-AI collaboration through chat dialogue and grounded instructions. |
| 2021-2022 | MineRL BASALT | Fuzzy human-like tasks where simple binary success is hard to define. |
| 2022 | MineDojo | Scaling up with thousands of tasks and Minecraft knowledge resources. |
| OpenAI VPT | Video PreTraining | Internet-scale imitation learning via inferred action labels. |
| 2023 | STEVE-1 | Language-conditioned Minecraft agent combining behavior knowledge and language/visual goal conditioning. |

The trend:

```text
fixed task
    -> long-horizon task
    -> hierarchy + imitation
    -> internet-scale pretraining
    -> language-conditioned agents
    -> LLM planning
    -> code-as-action
    -> skill libraries
    -> retrieval memory
    -> multimodal/VLA agents
    -> multi-constraint instructions
    -> vague creative goals
    -> human-AI collaboration
```


### Anssi's Promised Links

Source note: Anssi sent this list after the Day 2 Minecraft lecture. The links are preserved here as a reading map; this notebook does not infer extra architectural details beyond the classroom notes unless stated elsewhere.

Minecraft environments:

| Environment | Link |
| --- | --- |
| Malmo, first Minecraft env | https://www.microsoft.com/en-us/research/project/project-malmo/ |
| MineRL, more Minecraft-like environment | https://arxiv.org/abs/1907.13440 and https://github.com/minerllabs/minerl |
| CraftAssist | https://arxiv.org/abs/1907.08584 |
| MineRL BASALT, fuzzy tasks | https://arxiv.org/abs/2312.02405 and https://arxiv.org/abs/2303.13512 |
| MineDojo, extension of MineRL plus MineCLIP model | https://minedojo.org/ |

Minecraft agents, roughly chronological:

| Agent / system | Link |
| --- | --- |
| OpenAI VPT, large behavioural cloning | https://github.com/openai/Video-Pre-Training |
| STEVE-1, language-conditioned model | https://sites.google.com/view/steve-1 |
| Ghost in the Minecraft, LLM plus coded agents | https://arxiv.org/abs/2305.17144 |
| Voyager, code-generation policy | https://arxiv.org/abs/2305.16291 |
| Plan4MC | https://arxiv.org/abs/2303.16563 |
| JARVIS-1 | https://arxiv.org/abs/2311.05997 |
| OmniJARVIS | https://arxiv.org/abs/2407.00114 |
| DEPS | https://proceedings.neurips.cc/paper_files/paper/2023/file/6b8dfb8c0c12e6fafc6c256cb08a5ca7-Paper-Conference.pdf |
| MP5 | https://arxiv.org/abs/2312.07472v2 |
| Pan-1, goal-conditioned generalist agent | https://pantograph.com/journal/pan-1 |
| Creative Agents, fuzzy tasks | https://arxiv.org/abs/2312.02519 |
| Luban, fuzzy tasks | https://arxiv.org/abs/2405.15414 |


## 11. Minecraft Environments

### 2016 - Project Malmo

- early Minecraft environment for AI/RL agents,
- Python interface to launch/control a Minecraft player,
- used for maze solving and simple RL experiments.

### 2019 - MineRL

- more Minecraft-like tasks,
- long-horizon item dependency tree,
- example dependency chain:

```text
wood -> planks -> crafting table -> pickaxe -> stone -> ...
```

Important lesson:

- obtaining diamond was extremely difficult,
- strongest approaches relied heavily on hierarchy/task decomposition plus imitation,
- pure RL struggled with sparse reward and long horizons.

### 2019 - CraftAssist

- human-AI collaboration through chat dialogue,
- natural-language grounding,
- instruction following,
- collaborative building.

### 2021-2022 - MineRL BASALT

- fuzzy/human-like tasks,
- example: "build a waterfall",
- no simple binary success definition,
- human preference becomes important,
- top agents were rarely preferred to humans,
- many successful systems still relied on scripted behavior.

### 2022 - MineDojo

- "scaling things up",
- benchmark with thousands of tasks,
- uses Minecraft knowledge from YouTube, Wiki, and internet sources,
- includes MineCLIP for text-image correlation and semantic goal alignment.


### 中文导读 / Beginner Notes

Minecraft environments 的 evolution 可以理解成：

```text
Malmo: 先让 agent 能进 Minecraft
    -> MineRL: 更像真实 Minecraft，长任务更难
    -> BASALT: 任务变 fuzzy，需要 human preference
    -> MineDojo: 规模变大，任务和知识来源更多
```

**专业理解：** 环境越来越接近 open-ended embodied benchmark，不只是固定 reward game。


### OpenAI VPT - Video PreTraining

Problem:

- internet Minecraft videos contain observations,
- but they usually do not contain keyboard/mouse action labels.

VPT pipeline:

```text
small labelled dataset
    -> train inverse dynamics model (IDM)
    -> infer actions for large unlabeled video datasets
    -> behavioral clone policy from pseudo-labelled video
```

Concept:

```text
internet-scale imitation learning
    instead of
pure RL exploration from scratch
```

Why it matters:

- long-horizon Minecraft exploration is too sparse for naive RL,
- imitation gives the agent a prior over useful human-like behavior.

Failure mode:

- if the IDM labels actions incorrectly, errors become pseudo-labels for behavioral cloning.


### 2023 - STEVE-1

STEVE-1 is described in the classroom notes as a language-conditioned Minecraft agent.

It combines:

- VPT-style behavior knowledge,
- MineCLIP-style language/visual goal conditioning.

Conceptual shift:

```text
knows how to play
    ->
acts according to language goals
```


## 12. Minecraft Agents

This section follows the classroom sequence. When only a name/reference was provided, the note says so explicitly.

### Ghost in the Minecraft - Structured Skill Interface

Architecture:

```text
LLM planner
    -> choose structured action
    -> low-level interface executes keyboard/mouse behavior
```

Example structured actions:

- explore,
- mine,
- craft/smelt,
- dig_down.

Advantage:

- reduces the action space,
- makes high-level planning easier.

Limitation:

- pre-curated programmed skills limit capability.


### 中文导读 / Beginner Notes

Minecraft agents 的 evolution 可以理解成：

```text
imitation from video
    -> language-conditioned policy
    -> LLM planner + coded skills
    -> code generation as action
    -> skill library + memory
    -> multimodal/VLA generalist agents
    -> creative/fuzzy task agents
```

**Example:** Voyager-style agent 不只是选择“left/right/jump”，而是写一段程序去完成任务；这叫 code-as-action。


### Voyager-style Architecture - Code Generation as Action

Instead of manually coding every skill:

```text
new task
    -> LLM generates JavaScript / Mineflayer program
    -> execute code
    -> environment returns errors or success
    -> refine program
    -> successful skill stored
```

Key idea:

> program code itself is the action.

Why this is powerful:

- one generated program can contain many primitive actions,
- code gives temporal abstraction,
- execution errors provide concrete feedback,
- successful programs become reusable skills.


### Plan4MC, JARVIS-1, Pan-1

Source note: these names were listed in the Day 2 classroom context. Anssi's promised link list provides references for them, but detailed architectures were not covered in the supplied live notes here.

Therefore:

- **Plan4MC:** reference provided, detailed architecture not covered in the live notes.
- **JARVIS-1:** reference provided, detailed architecture not covered in the live notes.
- **Pan-1:** reference provided as a goal-conditioned generalist agent, detailed architecture not covered in the live notes.

The safe takeaway is that they belong to the broader family of Minecraft agents exploring planning, language grounding, multimodal control, memory, and long-horizon task execution.


### DEPS

DEPS highlights two problems with LLM planning in Minecraft:

1. More steps create more opportunities for failure.
2. Minecraft is open-ended, so there may be multiple valid solutions.

Therefore robust agents should use:

```text
plan -> act -> observe -> verify -> replan
```

not:

```text
plan once -> blindly execute
```

Why it matters:

- long-horizon plans fail locally,
- open worlds require adaptation,
- verification turns planning into a closed-loop process.


### MP5-style Memory / Subobjective Decomposition

The classroom notes connect MP5-style systems to:

- memory,
- subobjective decomposition,
- reusing previous solutions.

Conceptual pattern:

```text
long-horizon goal
    -> decompose into subobjectives
    -> solve or retrieve subgoal behavior
    -> compose toward final goal
```

This is close to retrieval-augmented agency: the retrieved object is not just text knowledge but a behavior, plan, code snippet, skill, or trajectory.


### OmniJARVIS / Unified Multimodal Policy

Alternative to modular planner-plus-skills:

```text
instruction + visual observation + history
    -> multimodal transformer
    -> action
```

It can handle:

- text instruction,
- observations,
- memory/context,
- action prediction.

Advantage:

- fewer manually engineered interfaces.

Challenge:

- the learning problem is harder because one model must bind perception, language, memory, and action.


### Creative Agents and Luban

These systems belong to the move from explicit tasks to vague creative objectives.

Architecture:

```text
vague language goal
    -> LLM clarification/decomposition
    -> design
    -> code/action generation
    -> environment
    -> VLM + human evaluation
```

Vision-language models can:

- select promising designs,
- judge visual semantics,
- check functionality,
- support iterative refinement.

Limitation:

- a VLM score is still a proxy, not identical to human preference.


## 13. LLM Planning and Hierarchical Execution

Minecraft illustrates why long-horizon planning matters.

Example goal:

```text
obtain stone pickaxe
```

Decomposition:

```text
get wood
    -> craft planks
    -> build crafting table
    -> make wooden pickaxe
    -> mine cobblestone
    -> make stone pickaxe
```

General architecture:

```text
long-horizon goal
    -> LLM task decomposition
    -> executable subtasks
    -> existing low-level agents/skills
    -> long-term completion
```

Why LLMs are useful:

- they have strong semantic knowledge,
- they can break vague goals into concrete steps,
- they can use textual knowledge such as recipes and Wiki-like facts.

Why LLM plans are risky:

- hallucinated dependencies,
- missing prerequisites,
- brittle assumptions,
- no guarantee that a textual plan is executable in the current world state.


### 中文导读 / Beginner Notes

Hierarchical execution 就是“先拆任务，再执行子任务”。

**Example:** obtain stone pickaxe:

```text
wood -> planks -> crafting table -> wooden pickaxe -> cobblestone -> stone pickaxe
```

**Common mistake:** LLM plan 不是一定正确。真实 agent 需要 observe/verify/replan。


## 14. Memory and Skill Reuse

### Skill Library / Procedural Memory

Successful code or skills can be stored:

- mine wood,
- craft table,
- craft stone sword,
- use furnace,
- make shield,
- fight zombie.

Distinction:

| Memory type | Meaning | Minecraft example |
| --- | --- | --- |
| Declarative memory | knowing what | "A stone pickaxe requires cobblestone and sticks." |
| Procedural memory | knowing how | executable routine to collect wood and craft tools |

Skill reuse turns one solved subtask into a reusable building block for future tasks.


### 中文导读 / Beginner Notes

记忆不是只记事实，也可以记“怎么做”。

- declarative memory: 知道 recipe 是什么
- procedural memory: 有一段可执行程序真的会做

**Example:** 不是只知道 "crafting table needs planks"，而是有一个 `craft_table()` skill 可以直接执行。


### Vector-database Memory

Retrieval loop:

```text
new subgoal
    -> embedding
    -> semantic search
    -> retrieve similar solved experience
    -> reuse/adapt solution
```

This is similar to RAG, but with a crucial difference:

- classic RAG retrieves documents or facts,
- retrieval-augmented agency can retrieve behavior, skills, plans, programs, or trajectories.

Why it matters:

- agents do not need to solve every subproblem from scratch,
- memory supports cumulative competence,
- semantic retrieval helps when tasks are phrased differently but require similar behavior.


## 15. VLM / VLA / Multimodal Agency

### VLM

Vision-Language Model:

```text
image/video + text
    -> description, judgment, answer, or score
```

A VLM can understand or evaluate visual content, such as judging whether a Minecraft build resembles "a yellow concrete house with a roof and windows."

### VLA

VLA means:

```text
Vision - Language - Action
```

| Component | Role |
| --- | --- |
| Vision | current environment observation |
| Language | human instruction |
| Action | environment control |

Contrast:

- VLM understands/evaluates.
- VLA acts.

This is central for embodied agents: the agent must connect what it sees, what it is asked to do, and how it controls the world.


### 中文导读 / Beginner Notes

VLM 和 VLA 最容易混：

- VLM: 看图 + 读文字，然后理解/评价
- VLA: 看图 + 读文字，然后行动

**Example:** VLM 能判断房子像不像 medieval fortress；VLA 要真的在 Minecraft 里移动、放方块、建出来。


## 16. From Explicit Tasks to Creative Goals

Explicit task:

```text
mine two logs
```

Multi-part constrained task:

```text
dig a block of sand near water at night with a wooden shovel
```

This includes:

- prerequisite constraints,
- object constraint,
- spatial constraint,
- temporal constraint,
- tool constraint.

Vague creative tasks:

- "build a yellow concrete house with a roof and windows",
- "build a medieval-inspired fortress",
- "build a sandstone palace with intricate details and towering minarets".

Why creative tasks are hard:

- no single correct answer,
- many valid designs,
- evaluation depends on human preference,
- VLM/human scoring becomes a proxy,
- Goodharting can happen if the proxy is overoptimized.


### 中文导读 / Beginner Notes

explicit task 有清楚成功条件；creative/fuzzy goal 没有唯一答案。

**Example:** "mine two logs" 很明确；"build a beautiful castle" 就需要审美、约束、迭代评价。

**Professional point:** fuzzy tasks push evaluation from binary reward toward human/VLM preference, which brings proxy-objective risk.


## 17. Minecraft as an AGI / Human-AI Collaboration Benchmark

The lecturer's final framing:

> A(G)I should at least complete tasks many human players can complete.

Fairness constraint:

- use similar input/output channels and resources as humans,
- visual environment,
- normal controls,
- internet,
- wiki.

Example benchmark tasks:

- defeat Ender Dragon in 24h,
- do it without dying,
- do it within 1h,
- obtain Elytra,
- obtain 1000 gunpowder in 24h.

The 1000-gunpowder task may require:

- understanding mechanics,
- planning,
- building farms,
- efficiency,
- long-horizon execution.

Preferred direction from the lecture:

```text
connect human players with AI
```

Research questions:

- How do humans use AI?
- How does AI react to humans?
- Does AI understand human intention?
- Which capabilities actually matter?

Final framing:

> Minecraft should not only test autonomous competence. It can test collaborative embodied agency.


### 中文导读 / Beginner Notes

这里的重点不是宣称 Minecraft 等于 AGI，而是说：如果一个系统连很多人类玩家能完成的 Minecraft 任务都做不了，那它的 embodied agency 还很有限。

**Example:** 24 小时内获得 1000 gunpowder 需要理解 mechanics、计划 farm、执行长任务，还可能需要和人协作。


## 18. Multi-Agent Reinforcement Learning

Single-agent RL assumes one learning agent interacting with an environment. The MARL lecture starts by pointing out why this is often insufficient:

- single-player Atari can fit a single-agent model,
- modern games are often multiplayer,
- real-world environments often contain other decision makers.

Example from the slides:

- a robot vacuum should not necessarily treat a human operator as static environment dynamics,
- the human can reasonably be modeled as another cooperative agent.

Basic MARL loop:

```text
agent 1 action
agent 2 action
...
agent n action
    -> joint action
    -> environment transition
    -> individual observations + rewards
    -> repeat
```

Formal intuition:

$$
\mathbf{a}_t = (a_t^1, a_t^2, \ldots, a_t^n)
$$

The environment transitions based on the **joint action**, not only one agent's action.


### 中文导读 / Beginner Notes

MARL 的难点是：你不是一个人在玩。其他 agent 也会行动、学习、合作或对抗。

**Example:** Boxing 不是静态环境。对手会躲、会打、会改变节奏。你的动作效果取决于对手同时做了什么。

**Common mistake:** 不要把其他 agent 简单当作固定背景；它们是 decision makers。


### Level-based Foraging Example

The slides use a level-based foraging task.

Setup:

- 3 robots,
- each robot has 6 possible actions:

```text
{up, down, left, right, collect, noop}
```

If one super-agent controls all three robots:

$$
6^3 = 216
$$

joint actions.

**Action-space explosion:** adding agents multiplies the number of joint actions.

Separate-agent solution:

- each robot is controlled by its own RL agent,
- but each agent's environment becomes non-stationary because the other learners change over time,
- coordination and credit assignment become harder.


In [4]:
actions_per_robot = 6
robots = 3
joint_actions = actions_per_robot ** robots
print(f"{actions_per_robot}^{robots} = {joint_actions} joint actions")

for n_agents in range(1, 7):
    print(f"{n_agents} agents with 6 actions each -> {6 ** n_agents:5d} joint actions")


6^3 = 216 joint actions
1 agents with 6 actions each ->     6 joint actions
2 agents with 6 actions each ->    36 joint actions
3 agents with 6 actions each ->   216 joint actions
4 agents with 6 actions each ->  1296 joint actions
5 agents with 6 actions each ->  7776 joint actions
6 agents with 6 actions each -> 46656 joint actions


### Dimensions of MARL Problems

The MARL lecture lists several dimensions:

| Dimension | Questions |
| --- | --- |
| Size | How many agents? Fixed or changing? How many states/actions? Discrete or continuous? |
| Knowledge | Do agents know their own/others' actions, rewards, transitions? |
| Observability | Full state, partial/noisy observations, other agents' actions/rewards? |
| Rewards | Zero-sum, common reward, or mixed/general-sum? |
| Objective | Learn equilibrium? Perform well while learning? Beat certain opponents? |
| Centralization and communication | Central controller? Independent policies? Shared information? Reliable communication? |

Reward structures:

| Structure | Meaning | Example |
| --- | --- | --- |
| Zero-sum | agents' rewards sum to zero | chess, boxing as pure win/loss abstraction |
| Common-reward | all agents share same reward | cooperative foraging |
| General-sum | arbitrary reward relationships | Diplomacy-like cooperation and competition |

Clarification: standard zero-sum means $\sum_i r_i = 0$. If a slide extraction appears to say rewards "sum to one," treat that as a source transcription/normalization issue rather than the usual definition.


## 19. Game Models

Model hierarchy:

| Model | Agents? | State? | Observability |
| --- | --- | --- | --- |
| Markov Chain | no decision-making agent | yes | full state |
| HMM | no decision-making agent | yes | partial/noisy observation |
| MDP | one agent | yes | full state |
| POMDP | one agent | yes | partial/noisy observation |
| Normal-form game | multiple agents | no environment state | one interaction |
| Repeated normal-form game | multiple agents | no environment state | action history matters |
| Stochastic / Markov game | multiple agents | yes | full state |
| POSG | multiple agents | yes | partial/noisy observations |
| Dec-POMDP | multiple agents | yes | common reward POSG |

The important shift from MDP to stochastic game:

```text
single action a_t
    ->
joint action (a_t^1, ..., a_t^n)
```


### 中文导读 / Beginner Notes

Game model evolution:

```text
normal-form game: 一次互动
    -> repeated game: 多轮互动，有历史
    -> stochastic game: 加入 state transition
    -> POSG: state 看不全
    -> Dec-POMDP: POSG + team common reward
```

**Example:** 石头剪刀布是 normal-form；有地图、有状态、有多名玩家的游戏更像 stochastic game/POSG。


### Normal-form Games

A normal-form game defines a single interaction between two or more agents.

Procedure from the slides:

1. Each agent selects its policy.
2. Each agent samples an action from its policy.
3. Actions form a joint action:

$$
\mathbf{a} = (a_1,\ldots,a_n)
$$

4. Each agent receives a reward based on the reward function and joint action.

For two agents, this can be represented as a matrix game.

Limitation:

- there is no evolving environment state,
- useful for studying agent-agent interaction but not rich sequential worlds.


### Repeated Normal-form Games

A normal-form game has one round. A repeated normal-form game has:

- finite $T$ rounds, or
- infinitely many rounds.

Policies may condition on action history:

$$
h_t = (a_0^1,\ldots,a_0^n,\ldots,a_{t-1}^1,\ldots,a_{t-1}^n)
$$

Why it matters:

- agents can punish, reward, cooperate, defect, or build reputation across rounds,
- history makes strategy richer.


### Stochastic Games

A stochastic game is close to an MDP but with multiple agents.

Clarification: a common tuple is:

$$
\mathcal{G} =
(N,\mathcal{S},\{\mathcal{A}_i\}_{i=1}^N,P,\{R_i\}_{i=1}^N,\gamma)
$$

At time $t$:

1. The game starts in state $s_t$.
2. Each agent observes the current state.
3. Each agent chooses an action based on its policy.
4. The joint action changes the state through $P$.
5. Each agent receives its own reward.

Transition:

$$
P(s_{t+1} \mid s_t, a_t^1,\ldots,a_t^N)
$$

This is the fully observed multi-agent analogue of an MDP.


### POSG and Dec-POMDP

POSG means Partially Observable Stochastic Game.

In a POSG:

- agents may not see the full state,
- observations can be partial or noisy,
- agents may not see other agents' actions,
- agents may not know other agents' rewards.

Examples:

- fog of war,
- card games,
- autonomous driving,
- limited local visual field.

A common-reward POSG is a Dec-POMDP:

```text
multiple agents
    + partial observability
    + shared reward
    -> Dec-POMDP
```


### Game Theory vs MARL

The lecture distinguishes classical game theory and MARL.

Classical game theory often assumes complete knowledge of:

- game rules,
- reward functions,
- transition model,
- observation structure.

MARL agents often do **not** know:

- other agents' reward functions,
- their own reward mapping explicitly in advance,
- transition dynamics,
- observation model,
- other agents' policies.

They learn from interaction.

Therefore:

```text
MARL problem = game model + solution concept
```


## 20. MARL Solution Concepts

### Expected Return

For agent $i$:

$$
J_i(\pi_i,\pi_{-i}) =
\mathbb{E}\left[\sum_{t=0}^{\infty}\gamma^t r_t^i\right]
$$

where:

- $\pi_i$ is agent $i$'s policy,
- $\pi_{-i}$ means the policies of all other agents,
- $r_t^i$ is agent $i$'s reward.


### 中文导读 / Beginner Notes

Solution concept 是“我们到底把什么叫作解”。

- best response: 别人不变，我怎么最好
- minimax: 最坏对手下我也尽量好
- Nash equilibrium: 每个人都没有单方面改变的动力

**Common mistake:** Nash equilibrium 不等于全体收益最大。它只是“没人愿意单独改”。


### Best Response

A best response holds other agents fixed and optimizes one agent:

$$
BR_i(\pi_{-i}) =
\arg\max_{\pi_i} J_i(\pi_i,\pi_{-i})
$$

Intuition:

- "If everyone else keeps doing what they are doing, what should I do?"

Limitation:

- if other agents are learning too, the target keeps moving.


### Minimax

Minimax is central for two-agent zero-sum games.

$$
\pi_i^* =
\arg\max_{\pi_i}
\min_{\pi_j}
J_i(\pi_i,\pi_j)
$$

Intuition:

- choose the policy that performs best against the worst-case opponent.

Why it matters:

- useful for adversarial games,
- robust to a strong opponent,
- can be conservative if the opponent is not actually worst-case.


### Nash Equilibrium

A Nash equilibrium is a joint policy where every agent is a best response to the others:

$$
J_i(\pi_i^*,\pi_{-i}^*) \ge
J_i(\pi_i,\pi_{-i}^*)
\quad \forall i,\ \forall \pi_i
$$

Intuition:

- no agent can improve by unilaterally changing policy while all others stay fixed.

Caveats emphasized in the lecture:

- equilibrium does not necessarily mean maximum possible expected return,
- equilibria may not be unique,
- there can be infinitely many equilibria.


## 21. Connections Across the Whole Day

The strongest connection is that every topic is about sequential decision making under imperfect feedback.

| Topic | State/context | Action | Reward/evaluation | Main difficulty |
| --- | --- | --- | --- | --- |
| Classical RL | environment state | primitive action | scalar reward | delayed consequences |
| DQN | image/feature state | discrete game action | Bellman target | unstable function approximation |
| MARL | state plus other agents | joint action component | individual/team reward | non-stationarity and coordination |
| LLM RL | prompt + generated context | token/response/tool call | preference/verifier reward | proxy mismatch and credit assignment |
| Agentic AI | task state + memory | tool call/code/subgoal | task completion | planning and verification |
| Minecraft | embodied world observation | control/program/subtask | sparse/fuzzy/human reward | long horizon and open-endedness |

Conceptual progression:

```text
Q-learning / DQN
    -> policy optimization
    -> RLHF / PPO / GRPO
    -> reasoning / verifier rewards
    -> hierarchical tool use
    -> embodied agents
    -> human-AI collaboration
```


### 中文导读 / Beginner Notes

全 Day 2 的主线是三个词：

- credit assignment: 成败该归因给哪一步？
- exploration: 怎么试出没见过但更好的行为？
- evaluation: 怎么知道优化目标没有骗我们？

**Evolution view:** 方法越强，action 越高级，reward 越复杂，evaluation 也越难。


### Credit Assignment

Credit assignment appears everywhere:

| Setting | Credit assignment question |
| --- | --- |
| DQN | Which action caused later reward? |
| Minecraft | Which subtask failure caused overall failure? |
| LLM reasoning | Which reasoning token/step caused final correctness? |
| MARL | Which agent contributed to team reward? |
| Hierarchical agents | Was failure caused by manager, worker, tool, memory, or verifier? |

This explains why process supervision, hierarchy, replay buffers, target networks, group-relative rewards, and verifiers all matter: they are different attempts to make learning signals more usable.


### Exploration

Exploration also repeats across Day 2:

| Setting | Exploration form |
| --- | --- |
| DQN | epsilon-greedy random action |
| PPO/GRPO | sampled policy trajectories |
| LLM reasoning | multiple candidate reasoning paths |
| Minecraft | world exploration and subtask discovery |
| MARL | opponent-dependent and teammate-dependent exploration |

Too little exploration:

- agent stays inside training distribution,
- no new strategies are discovered,
- imitation or preference learning can become narrow.

Too much exploration:

- inefficient,
- unsafe,
- unstable,
- expensive for large models.


### Evaluation / Reward Design

Day 2 repeatedly shows that reward is not the same as the true objective.

| Proxy | Risk |
| --- | --- |
| scalar game reward | reward hacking or boring exploit |
| human preference | disagreement, cost, bias |
| AI judge | amplified evaluator bias |
| process reward model | reasoning hacking |
| unit tests | incomplete specification gaming |
| VLM score | visual proxy mismatch with human preference |
| win/loss | hides how robust or enjoyable the agent is |

The future direction is dynamic evaluation:

```text
interaction + verifier + real task completion
```


## 22. Common Confusions

| Confusion | Clean distinction |
| --- | --- |
| policy vs Q-function | A policy chooses actions. A Q-function evaluates actions. |
| reward vs return | Reward is immediate. Return is cumulative discounted future reward. |
| value vs Q-value | $V(s)$ values a state. $Q(s,a)$ values an action in a state. |
| PPO vs GRPO | PPO uses policy/reference/reward/value components; GRPO removes separate critic/value and uses group-relative rewards. |
| RLHF vs DPO | RLHF trains an explicit reward model then optimizes policy; DPO directly optimizes from preference pairs. |
| RLAIF vs Constitutional AI | RLAIF uses AI feedback broadly; Constitutional AI uses explicit principles/rules for critique and revision. |
| outcome vs process supervision | Outcome rewards final answer; process supervision rewards intermediate steps. |
| verifier-guided RL vs RLVR | Verifier-guided may use learned verifier; RLVR emphasizes deterministic/objective verifiable rewards. |
| VLM vs VLA | VLM understands/evaluates vision+language; VLA acts from vision+language. |
| planner vs worker | Planner chooses subgoals/tools; worker executes low-level behavior. |
| skill library vs vector memory | Skill library stores executable skills; vector memory retrieves semantically similar experiences/skills/plans. |
| MDP vs stochastic game vs POSG | MDP is single-agent full state; stochastic game is multi-agent full state; POSG is multi-agent partial observation. |
| zero-sum vs common-reward vs general-sum | Zero-sum is pure competition; common reward is team cooperation; general-sum can mix competition and cooperation. |
| best response vs Nash equilibrium | Best response optimizes against fixed others; Nash means everyone is simultaneously best responding. |


### 中文导读 / Beginner Notes

这一节是考试和写报告最容易出错的地方。建议把每一行都能用自己的例子解释出来。

**Example:** 如果你能用 Boxing 解释 policy vs Q-function，再用 Minecraft 解释 planner vs worker，说明你真的理解了，而不是只背表格。


## 23. Key Formulas Cheat Sheet

Discounted return:

$$
G_t = \sum_{k=0}^{\infty}\gamma^k r_{t+k}
$$

State value:

$$
V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t=s]
$$

Action value:

$$
Q^\pi(s,a) = \mathbb{E}_\pi[G_t \mid s_t=s,a_t=a]
$$

Bellman optimality target:

$$
y = r + \gamma \max_{a'}Q(s',a')
$$

Q-learning update:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha(y - Q(s,a))
$$

DQN loss:

$$
L(\theta)=
\frac{1}{N}\sum_i
(y_i - Q_\theta(s_i,a_i))^2
$$

PPO ratio:

$$
r_t(\theta)=
\frac{\pi_\theta(a_t\mid s_t)}
     {\pi_{\theta_{\text{old}}}(a_t\mid s_t)}
$$

Group-relative advantage:

$$
\hat{A}_i =
\frac{r_i-\mu_R}{\sigma_R+\epsilon}
$$

Joint action:

$$
\mathbf{a}_t = (a_t^1,\ldots,a_t^n)
$$

Agent expected return:

$$
J_i(\pi_i,\pi_{-i}) =
\mathbb{E}\left[\sum_{t=0}^{\infty}\gamma^t r_t^i\right]
$$

Best response:

$$
BR_i(\pi_{-i}) =
\arg\max_{\pi_i}J_i(\pi_i,\pi_{-i})
$$

Minimax:

$$
\pi_i^* =
\arg\max_{\pi_i}\min_{\pi_j}J_i(\pi_i,\pi_j)
$$

Nash equilibrium:

$$
J_i(\pi_i^*,\pi_{-i}^*) \ge
J_i(\pi_i,\pi_{-i}^*)
$$


### 中文导读 / Beginner Notes

公式读法：

- 先看符号在说什么，不要先怕数学。
- $r$ 是现在，$\gamma \max Q(s',a')$ 是未来。
- 多 agent 时，把单个 action 换成 joint action。

**Example:** DQN target 就是“这一步奖励 + 下一局面最好动作的估计价值”。


## 24. Day 2 Final Summary

Day 2 starts from standard RL and expands the idea of "action" step by step.

In DQN, an action is a discrete game move and the agent learns $Q(s,a)$ from replayed transitions.

In RL for LLMs, an action may be a token, response, or tool call. RL moves optimization from next-token likelihood toward sequence-level objectives such as preference, correctness, or task completion.

In agency, actions become higher-level operations: plans, tools, programs, subgoals, and revisions.

In Minecraft, these ideas become embodied. The agent must perceive, plan, craft, explore, remember, execute, and adapt in an open-ended environment.

In MARL, the environment includes other agents. The core transition depends on a joint action, and learning must handle coordination, competition, observability, and equilibrium concepts.

The main Day 2 thesis:

> The hard part is not only choosing the next action. It is choosing actions whose delayed, strategic, and proxy-measured consequences actually match the intended goal.


### 中文导读 / Beginner Notes

一句话总结：

> Day 2 讲的是 action 逐渐变复杂：从按钮，到 token，到工具调用，到程序，到多 agent 联合动作。

你要带走的不只是术语，而是 evolution：为什么旧方法不够，所以才出现新方法。


## 25. Self-test Questions

### Short Conceptual Questions

1. What is the difference between reward and return?
2. What does $Q(s,a)$ estimate?
3. Why does DQN use a replay buffer?
4. What is epsilon-greedy exploration?
5. Why is next-token prediction not the same as task success?
6. Name the four major components in PPO-style RLHF from the lecture.
7. What does GRPO remove compared with PPO?
8. What is the main risk of RLAIF?
9. What is the critique-revision gap in Constitutional AI?
10. What does DPO bypass?
11. Why is outcome-based RL sparse?
12. Give one example of an RLVR verifier.
13. What does VLA stand for?
14. Why is MineRL difficult for pure RL?
15. What is a joint action in MARL?

### Medium Questions

1. Explain how the DQN Bellman target is different for terminal and non-terminal transitions.
2. Compare RLHF, RLAIF, Constitutional AI, DPO, and UNA as alignment methods.
3. Compare outcome supervision and process supervision for reasoning.
4. Explain why a learned verifier can be exploited.
5. Describe the VPT pipeline and why the inverse dynamics model is needed.
6. Explain the difference between declarative and procedural memory in Minecraft agents.
7. Explain how vector-database memory supports retrieval-augmented agency.
8. Explain why a super-agent controlling three robots with six actions each has 216 joint actions.
9. Distinguish normal-form game, repeated normal-form game, stochastic game, and POSG.
10. Explain best response and Nash equilibrium in your own words.

### Harder Synthesis Questions

1. How does credit assignment appear in DQN, Minecraft, LLM reasoning, MARL, and hierarchical agents?
2. Why can optimizing a reward model too strongly reduce true usefulness or correctness?
3. Design a Minecraft agent loop for "obtain 1000 gunpowder in 24h" using planning, memory, verification, and replanning.
4. Compare DQN exploration, GRPO candidate sampling, Minecraft world exploration, and MARL opponent-dependent exploration.
5. Why is Minecraft useful not only for autonomous competence but also for human-AI collaboration?


### 中文导读 / Beginner Notes

自测建议：

1. 先不用看答案，尝试用中英混合解释。
2. 每题都加一个游戏例子，比如 Boxing/Minecraft/Diplomacy。
3. 如果只能背定义但举不出例子，说明还需要回到对应章节。


<details>
<summary><strong>Answer Key</strong></summary>

### Short Answers

1. Reward is immediate feedback; return is cumulative discounted future reward.
2. $Q(s,a)$ estimates expected return after taking action $a$ in state $s$ and then following a policy.
3. Replay buffers decorrelate sequential data, reuse samples, and stabilize mini-batch training.
4. With probability $\epsilon$ choose random action; otherwise choose the greedy best action.
5. Next-token prediction optimizes local likelihood, not full-response correctness, safety, or task completion.
6. Policy, reference model, reward model, and value/critic.
7. GRPO removes the separate critic/value model and uses group-relative reward statistics.
8. Systematic evaluator bias can be amplified.
9. The model may critique correctly but fail to revise correctly.
10. DPO bypasses an explicit reward model.
11. It usually gives reward only for the final outcome.
12. Compiler, unit tests, symbolic solver, proof checker, or game-engine success check.
13. Vision-Language-Action.
14. Sparse reward and long-horizon item dependencies make random exploration inefficient.
15. A tuple of all agents' simultaneous actions.

### Medium Answers

1. Non-terminal target is $r+\gamma\max_{a'}Q(s',a')$; terminal target is just $r$.
2. RLHF uses human rankings and reward model; RLAIF uses AI feedback; Constitutional AI uses explicit principles for critique/revision; DPO learns directly from preference pairs; UNA unifies multiple feedback types through an implicit reward/regression framing.
3. Outcome supervision is cheaper and sparse but weak for credit assignment; process supervision is denser and better for intermediate reasoning but costly and vulnerable to reasoning hacking.
4. If the verifier is a learned proxy, the policy can discover outputs that score highly without satisfying the true objective.
5. VPT trains an IDM on labelled data, uses it to infer actions for unlabeled videos, then behavioral-clones from pseudo-labelled large-scale video.
6. Declarative memory stores facts; procedural memory stores how-to behavior or executable skills.
7. It embeds new subgoals, retrieves similar solved experiences, and adapts the stored plan/skill/code.
8. Each of 3 robots has 6 options, so the joint action count is $6 \times 6 \times 6 = 216$.
9. Normal-form is one interaction; repeated normal-form repeats interaction with history; stochastic game adds state transitions; POSG adds partial/noisy observations.
10. Best response optimizes one agent against fixed others; Nash equilibrium is mutual best response.

### Harder Answers

1. Credit assignment asks which earlier action/subtask/token/agent/manager decision caused later success or failure.
2. Strong optimization exploits proxy weaknesses: reward model shortcuts, sycophancy, verbosity, over-refusal, reasoning hacking, or specification gaming.
3. A strong loop decomposes the task, retrieves farm/build/combat skills, executes subtasks, checks inventory and environment state, detects failures, and replans rather than executing one static plan.
4. All are ways to sample alternatives under uncertainty, but they differ in action scale: primitive actions, generated responses, world navigation, and strategy against adaptive agents.
5. Minecraft is human-first, open-ended, well understood by players, and supports shared tasks where AI must interpret intent, communicate, assist, and adapt.

</details>
